# Tenrec QK-article：In-batch Softmax 双塔 + ANN 多路召回

**数据语义**：QK-article 的每一行是一次已点击的正反馈；它没有逐曝光的 `click=0` 行。因此双塔训练采用 batch 内其它正样本物品作为隐式负样本。

召回路：ItemCF、UserCF、标签召回、双塔 ANN。双塔在训练后用 FAISS HNSW 检索；四路候选随后去重合并。

In [ ]:
# 如缺少依赖，取消注释后执行一次。
# 镜像已包含与 CUDA 匹配的 torch；不要在这里重新安装 torch。
# %pip install -q pandas pyarrow scipy implicit tqdm faiss-cpu scikit-learn
from __future__ import annotations
import os, time
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from implicit.nearest_neighbours import BM25Recommender, CosineRecommender
from tqdm.auto import tqdm
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F
import faiss

RAW_DATA_PATH = Path(os.environ.get('TENREC_QK_ARTICLE_PATH', '/root/shared-storage/Tenrec_QK_article/Tenrec/QK-article.csv'))
PROJECT_ROOT = Path(os.environ.get('TENREC_PROJECT_ROOT', '/root/workspace/tenrec_qk_article_inbatch_ann'))
DATA_DIR, MODEL_DIR, RESULT_DIR = PROJECT_ROOT/'data', PROJECT_ROOT/'models', PROJECT_ROOT/'results'
for p in (DATA_DIR, MODEL_DIR, RESULT_DIR): p.mkdir(parents=True, exist_ok=True)

SEED, SAMPLE_USERS, MIN_POSITIVE_CLICKS = 2026, 100_000, 10
CHUNK_SIZE, EVAL_USERS, RECALL_K = 500_000, 2_000, 100
BATCH_SIZE, EPOCHS, LATENT_DIM, LEARNING_RATE, TEMPERATURE = 1024, 10, 64, 2e-3, 0.07
ANN_SEARCH_K, HNSW_M, HNSW_EF_SEARCH = 500, 32, 128
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
np.random.seed(SEED); torch.manual_seed(SEED)
print({'device': str(DEVICE), 'raw_path': str(RAW_DATA_PATH), 'temperature': TEMPERATURE})

## 1. 固定抽样 10 万用户并读取必要字段

先完整扫描 `user_id`，固定随机选用户；再分块只保留这些用户的字段。`event_order` 保留 CSV 原始顺序，作为论文所述的行为顺序近似。

In [ ]:
required_columns = ['user_id','item_id','click','like','share','follow','read','favorite','read_percentage','category_first','category_second','item_score1','item_score2','item_score3','read_time','gender','age']
assert RAW_DATA_PATH.exists(), f'找不到数据：{RAW_DATA_PATH}'
users = set()
for chunk in tqdm(pd.read_csv(RAW_DATA_PATH, usecols=['user_id'], chunksize=CHUNK_SIZE), desc='scan users'):
    users.update(chunk['user_id'].dropna().unique().tolist())
selected_users = set(np.random.default_rng(SEED).choice(np.array(list(users)), size=min(SAMPLE_USERS, len(users)), replace=False).tolist())
parts, event_offset = [], 0
for chunk in tqdm(pd.read_csv(RAW_DATA_PATH, usecols=required_columns, chunksize=CHUNK_SIZE, low_memory=False), desc='read selected users'):
    chosen = chunk[chunk['user_id'].isin(selected_users)].copy()
    chosen['event_order'] = np.arange(event_offset, event_offset + len(chosen), dtype=np.int64)
    event_offset += len(chosen)
    parts.append(chosen)
events = pd.concat(parts, ignore_index=True)
events.to_parquet(DATA_DIR/'qk_article_100k_events.parquet', index=False)
print({'rows': len(events), 'users': events.user_id.nunique(), 'items': events.item_id.nunique(), 'click_distribution': events.click.value_counts(dropna=False).to_dict()})
assert pd.to_numeric(events['click'], errors='coerce').fillna(0).gt(0).all(), '该 QK-article 文件应只含点击正样本；请检查输入文件。'

## 2. 切分、编码与三路非向量召回

按每位用户的行为顺序切为 8:1:1：前 80% 训练、中间 10% 验证、最后 10% 测试。所有统计只在训练集上建立，避免未来信息泄漏。

In [ ]:
positive = events.sort_values(['user_id','event_order'], kind='stable').copy()
n = positive.groupby('user_id')['item_id'].transform('size')
positive = positive[n >= MIN_POSITIVE_CLICKS].copy()
positive['position'] = positive.groupby('user_id').cumcount()
positive['n_user_events'] = positive.groupby('user_id')['item_id'].transform('size')
positive['train_end'] = np.floor(0.8 * positive.n_user_events).astype(int)
positive['val_end'] = np.floor(0.9 * positive.n_user_events).astype(int)
train_raw = positive[positive.position < positive.train_end].copy()
val_events = positive[(positive.position >= positive.train_end) & (positive.position < positive.val_end)].copy()
test_events = positive[positive.position >= positive.val_end].copy()
val_df = val_events[['user_id','item_id']].copy()
test_df = test_events[['user_id','item_id']].copy()
train_df = (train_raw.sort_values('event_order').groupby(['user_id','item_id'], as_index=False, sort=False)
            .agg(gender=('gender','last'), age=('age','last'), category_first=('category_first','last'), category_second=('category_second','last'),
                 item_score1=('item_score1','mean'), item_score2=('item_score2','mean'), item_score3=('item_score3','mean'),
                 like=('like','max'), share=('share','max'), follow=('follow','max'), favorite=('favorite','max'), read_time=('read_time','mean'), read_percentage=('read_percentage','mean'), event_order=('event_order','max')))
user_ids, item_ids = np.sort(train_df.user_id.unique()), np.sort(train_df.item_id.unique())
u2i, it2i = {x:i for i,x in enumerate(user_ids)}, {x:i for i,x in enumerate(item_ids)}
train_df = train_df[train_df.user_id.isin(u2i) & train_df.item_id.isin(it2i)].copy()
ui = csr_matrix((np.ones(len(train_df), dtype=np.float32), (train_df.user_id.map(u2i), train_df.item_id.map(it2i))), shape=(len(user_ids),len(item_ids)))
seen = {u:set(g.item_id) for u,g in train_df.groupby('user_id')}
eligible_eval = test_df[test_df.user_id.isin(u2i) & test_df.item_id.isin(it2i)].copy()
# 所有召回路都会过滤训练期已点击文章，因此测试目标也排除重复点击，保证评测口径一致。
eligible_eval = eligible_eval[[row.item_id not in seen.get(row.user_id,set()) for row in eligible_eval.itertuples()]]
eval_user_pool = eligible_eval.user_id.drop_duplicates().to_numpy()
eval_users = np.random.default_rng(SEED).choice(eval_user_pool, size=min(EVAL_USERS, len(eval_user_pool)), replace=False)
eval_targets = eligible_eval[eligible_eval.user_id.isin(eval_users)].groupby('user_id').item_id.apply(lambda s: set(s)).to_dict()
print({'train_pairs':len(train_df), 'two_tower_samples':len(train_df), 'validation_events':len(val_df), 'test_events':len(test_df), 'eval_users':len(eval_targets), 'eval_targets':sum(map(len,eval_targets.values())), 'matrix_shape':ui.shape, 'sparsity':1-ui.nnz/(ui.shape[0]*ui.shape[1])})

# implicit 的 KNN 模型计算输入矩阵“列”之间的相似度：ui 的列是 item，ui.T 的列是 user。
item_cf = BM25Recommender(K=200); item_cf.fit(ui.tocsr(), show_progress=True)
user_cf = CosineRecommender(K=100); user_cf.fit(ui.T.tocsr(), show_progress=True)
assert item_cf.similarity.shape[0] == len(item_ids), (item_cf.similarity.shape, len(item_ids))
assert user_cf.similarity.shape[0] == len(user_ids), (user_cf.similarity.shape, len(user_ids))
tag_col = 'category_second'
tag_items = train_df.dropna(subset=[tag_col]).groupby(tag_col).item_id.apply(lambda s: s.value_counts().index.to_list()).to_dict()
user_tags = train_df.dropna(subset=[tag_col]).groupby('user_id')[tag_col].apply(lambda s: s.value_counts().index.to_list()).to_dict()

def itemcf_recall(uid, k=RECALL_K):
    scores = {}
    for iid in seen.get(uid, set()):
        ids, sim = item_cf.similar_items(it2i[iid], N=80)
        for j, s in zip(ids, sim):
            cand = item_ids[j]
            if cand not in seen[uid]: scores[cand] = scores.get(cand, 0.) + float(s)
    return [x for x,_ in sorted(scores.items(), key=lambda z:z[1], reverse=True)[:k]]
def usercf_recall(uid, k=RECALL_K):
    ids, sim = user_cf.similar_items(u2i[uid], N=80); scores = {}
    for v, s in zip(ids, sim):
        if int(v) == u2i[uid]: continue
        for j in ui[int(v)].indices: scores[item_ids[j]] = scores.get(item_ids[j], 0.) + float(s)
    return [x for x,_ in sorted(scores.items(), key=lambda z:z[1], reverse=True) if x not in seen[uid]][:k]
def tag_recall(uid, k=RECALL_K):
    out=[]
    for tag in user_tags.get(uid, []): out.extend(tag_items.get(tag, []))
    return list(dict.fromkeys(x for x in out if x not in seen[uid]))[:k]

## 3. 双塔：In-batch Softmax（温度 0.07）

一个 batch 的第 $i$ 行是正对 $(u_i, i_i^+)$；同 batch 内其余 $N-1$ 个物品视为该用户的隐式负样本。

$$L=-\frac{1}{N}\sum_i\log\frac{\exp(\cos(u_i,i_i^+)/\tau)}{\sum_{j=1}^N\exp(\cos(u_i,i_j)/\tau)},\quad \tau=0.07$$

这不等同于真实曝光负样本：其中可能存在少量假负样本，属于 QK-article 仅正反馈数据的固有限制。

In [ ]:
def code_map(s): return {v:i+1 for i,v in enumerate(sorted(s.dropna().astype(str).unique()))}
gmap, amap = code_map(train_df.gender), code_map(train_df.age)
c1map, c2map = code_map(train_df.category_first), code_map(train_df.category_second)
print('1/3 类别特征编码完成', {'gender':len(gmap), 'age':len(amap), 'category_first':len(c1map), 'category_second':len(c2map)})
user_behavior = train_df.groupby('user_id',as_index=False,sort=False).agg(gender=('gender','last'),age=('age','last'),like_rate=('like','mean'),share_rate=('share','mean'),follow_rate=('follow','mean'),favorite_rate=('favorite','mean'),mean_read_time=('read_time','mean'),mean_read_percentage=('read_percentage','mean'))
uf = pd.DataFrame({'user_id':user_ids}).merge(user_behavior, on='user_id', how='left')
itf = pd.DataFrame({'item_id':item_ids}).merge(train_df.groupby('item_id',as_index=False,sort=False).agg(category_first=('category_first','last'),category_second=('category_second','last'),item_score1=('item_score1','mean'),item_score2=('item_score2','mean'),item_score3=('item_score3','mean')), on='item_id', how='left')
def mapped(s, mp): return torch.tensor(s.astype(str).map(mp).fillna(0).astype('int64').to_numpy(), device=DEVICE)
ug, ua = mapped(uf.gender,gmap), mapped(uf.age,amap)
user_numeric = uf[['like_rate','share_rate','follow_rate','favorite_rate','mean_read_time','mean_read_percentage']].apply(pd.to_numeric,errors='coerce').fillna(0.).to_numpy(dtype=np.float32, copy=True)
user_numeric[:,4] = np.log1p(np.clip(user_numeric[:,4], 0, None))
user_numeric = (user_numeric-user_numeric.mean(0))/(user_numeric.std(0)+1e-6); user_numeric=torch.tensor(user_numeric,device=DEVICE)
ic1, ic2 = mapped(itf.category_first,c1map), mapped(itf.category_second,c2map)
q = itf[['item_score1','item_score2','item_score3']].apply(pd.to_numeric,errors='coerce').fillna(0.).to_numpy(np.float32)
q = (q-q.mean(0))/(q.std(0)+1e-6); quality=torch.tensor(q, device=DEVICE)
print('2/3 用户与文章侧特征张量完成', {'users':len(user_ids), 'items':len(item_ids), 'device':str(DEVICE)})

class TwoTower(nn.Module):
    def __init__(self):
        super().__init__(); self.ue=nn.Embedding(len(user_ids),LATENT_DIM); self.ge=nn.Embedding(len(gmap)+1,8); self.ae=nn.Embedding(len(amap)+1,8)
        self.ie=nn.Embedding(len(item_ids),LATENT_DIM); self.c1e=nn.Embedding(len(c1map)+1,8); self.c2e=nn.Embedding(len(c2map)+1,8)
        self.um=nn.Sequential(nn.Linear(LATENT_DIM+22,128),nn.ReLU(),nn.Linear(128,LATENT_DIM))
        self.im=nn.Sequential(nn.Linear(LATENT_DIM+19,128),nn.ReLU(),nn.Linear(128,LATENT_DIM))
    def user(self,u): return F.normalize(self.um(torch.cat([self.ue(u),self.ge(ug[u]),self.ae(ua[u]),user_numeric[u]],-1)),dim=-1)
    def item(self,i): return F.normalize(self.im(torch.cat([self.ie(i),self.c1e(ic1[i]),self.c2e(ic2[i]),quality[i]],-1)),dim=-1)

# 每条样本是 (user, 正文章)；同 batch 内其余文章作为该用户的动态 N-1 个负样本。
tu, ti = train_df.user_id.map(u2i).to_numpy(np.int64), train_df.item_id.map(it2i).to_numpy(np.int64)
loader = DataLoader(TensorDataset(torch.from_numpy(tu),torch.from_numpy(ti)), batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=0, pin_memory=torch.cuda.is_available())
model=TwoTower().to(DEVICE); opt=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE)
print('3/3 开始双塔训练', {'samples':len(tu), 'batch_size':BATCH_SIZE, 'batches_per_epoch':len(loader), 'epochs':EPOCHS})
for epoch in range(EPOCHS):
    model.train(); total=0.; count=0
    progress=tqdm(loader, desc=f'epoch {epoch+1}/{EPOCHS}', mininterval=1.0)
    for step, (u, pos) in enumerate(progress, start=1):
        u,pos=u.to(DEVICE),pos.to(DEVICE); logits=(model.user(u) @ model.item(pos).T) / TEMPERATURE
        loss=F.cross_entropy(logits, torch.arange(len(u),device=DEVICE))
        opt.zero_grad(); loss.backward(); opt.step(); total += loss.item()*len(u); count += len(u)
        if step == 1 or step % 50 == 0: progress.set_postfix(batch_loss=f'{loss.item():.4f}', avg_loss=f'{total/count:.4f}')
    print(f'epoch={epoch+1}/{EPOCHS}, inbatch_ce={total/count:.5f}', flush=True)
torch.save(model.state_dict(), MODEL_DIR/'qk_article_inbatch_two_tower.pt')

## 4. Item 向量建 FAISS HNSW 索引，ANN 召回

HNSW 是近似最近邻索引；`efSearch` 越大越接近精确检索、但查询越慢。线上可持久化该索引并随着 item 更新重建/增量更新。

In [ ]:
@torch.no_grad()
def encode_all_items(batch=8192):
    model.eval(); return np.vstack([model.item(torch.arange(s,min(s+batch,len(item_ids)),device=DEVICE)).cpu().numpy() for s in range(0,len(item_ids),batch)]).astype('float32')
item_vectors = encode_all_items()
ann = faiss.IndexHNSWFlat(LATENT_DIM, HNSW_M, faiss.METRIC_INNER_PRODUCT)
ann.hnsw.efConstruction = 200; ann.hnsw.efSearch = HNSW_EF_SEARCH; ann.add(item_vectors)
faiss.write_index(ann, str(MODEL_DIR/'qk_article_hnsw.index'))
@torch.no_grad()
def tower_ann_recall(uid, k=RECALL_K):
    model.eval(); u=torch.tensor([u2i[uid]],device=DEVICE); vec=model.user(u).cpu().numpy().astype('float32')
    _, ids=ann.search(vec, ANN_SEARCH_K); candidates=[item_ids[j] for j in ids[0] if j >= 0 and item_ids[j] not in seen[uid]]
    return candidates[:k]
# 小批量测量 ANN 查询耗时（包括用户塔前向及已看过滤）。
probe=list(eval_targets)[:min(200,len(eval_targets))]; start=time.perf_counter(); _=[tower_ann_recall(u) for u in probe]
print({'ann_index_items':ann.ntotal, 'avg_query_ms':1000*(time.perf_counter()-start)/max(1,len(probe)), 'efSearch':HNSW_EF_SEARCH})

## 5. 多路合并与指标

报告两组结果：第一组将每个单路和等权 RRF 融合结果都限制为 Top-100，便于公平比较；第二组保留四路去重后的完整候选池，作为后续粗排输入。每个用户有多个测试正样本：Recall 为逐用户召回率的平均值，HitRate 表示至少命中一条测试正样本的用户比例。

In [ ]:
recallers={'itemcf':itemcf_recall, 'usercf':usercf_recall, 'tag':tag_recall, 'two_tower_ann':tower_ann_recall}
results={name:{u:fn(u) for u in tqdm(eval_targets,desc=name)} for name,fn in recallers.items()}

# 完整候选池：四路结果去重后全部保留，供后续粗排使用（通常约 200~400 条/用户）。
results['merged_pool']={u:list(dict.fromkeys(item for name in recallers for item in results[name][u])) for u in eval_targets}

# 公平的 Top-100 合并：等权 Reciprocal Rank Fusion。权重只能在验证集调，不能看测试结果后调。
RRF_K=60
SOURCE_WEIGHTS={name:1.0 for name in recallers}
def rrf_merge(uid, k=RECALL_K):
    scores={}
    for name in recallers:
        for rank, item in enumerate(results[name][uid], start=1):
            scores[item]=scores.get(item,0.0)+SOURCE_WEIGHTS[name]/(RRF_K+rank)
    return [item for item,_ in sorted(scores.items(),key=lambda x:x[1],reverse=True)[:k]]
results['merged_rrf']={u:rrf_merge(u) for u in eval_targets}
def metrics(candidates):
    per_user=[len(set(candidates[u]) & eval_targets[u])/len(eval_targets[u]) for u in eval_targets]
    hits=[int(bool(set(candidates[u]) & eval_targets[u])) for u in eval_targets]
    unique=set(x for xs in candidates.values() for x in xs)
    return {'recall@%d'%RECALL_K:float(np.mean(per_user)), 'hit_rate@%d'%RECALL_K:float(np.mean(hits)), 'coverage@%d'%RECALL_K:len(unique)/len(item_ids), 'avg_candidates':float(np.mean([len(x) for x in candidates.values()]))}
top100_names=list(recallers)+['merged_rrf']
report=pd.DataFrame({name:metrics(results[name]) for name in top100_names}).T.sort_values('recall@%d'%RECALL_K,ascending=False)
pool_raw=metrics(results['merged_pool'])
pool_report=pd.DataFrame([{'recall@pool':pool_raw['recall@%d'%RECALL_K], 'hit_rate@pool':pool_raw['hit_rate@%d'%RECALL_K], 'coverage@pool':pool_raw['coverage@%d'%RECALL_K], 'avg_candidates':pool_raw['avg_candidates']}],index=['merged_pool'])
print('同为 Top-100 的公平比较：')
display(report)
print('四路完整去重候选池（交给后续粗排）：')
display(pool_report)
pd.DataFrame([{'user_id':u,'test_target_items':sorted(eval_targets[u]),**{name:xs[u] for name,xs in results.items()}} for u in eval_targets]).to_parquet(RESULT_DIR/'qk_article_multirecall_candidates.parquet',index=False)
report.to_csv(RESULT_DIR/'qk_article_multirecall_top100_metrics.csv')
pool_report.to_csv(RESULT_DIR/'qk_article_multirecall_pool_metrics.csv')
print('saved to', RESULT_DIR)

## 6. 粗排/精排候选样本

QK-article 没有真实曝光未点击行，因此排序阶段把“召回但未出现在用户已知点击集合中的文章”作为**隐式负样本**。训练时会注入排序区间的正样本，验证与测试不注入。排序训练/验证用户从原 10% 验证区间拆分，最后 10% 测试区间保持不动。

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

RANK_SEED=2027
RANK_TOTAL_USERS=12_000       # 首轮实验；显存足够时可增大
RANK_TRAIN_USER_RATIO=0.8
NEGATIVES_PER_POSITIVE=20
RANK_BATCH_SIZE=4096
TEACHER_EPOCHS=5
STUDENT_EPOCHS=5
COARSE_TOPK=100
FINE_TOPK=20
SOURCE_ORDER=('itemcf','usercf','tag','two_tower_ann')
TASKS=('relevance','like','share','follow','favorite','long_read')

rank_source=val_events[val_events.user_id.isin(u2i) & val_events.item_id.isin(it2i)].copy()
rank_source=rank_source[[row.item_id not in seen.get(row.user_id,set()) for row in rank_source.itertuples()]]
rng=np.random.default_rng(RANK_SEED)
rank_user_pool=rng.permutation(rank_source.user_id.drop_duplicates().to_numpy(copy=True))
rank_user_pool=rank_user_pool[:min(RANK_TOTAL_USERS,len(rank_user_pool))]
n_rank_train=max(1,int(len(rank_user_pool)*RANK_TRAIN_USER_RATIO))
rank_train_users=rank_user_pool[:n_rank_train]; rank_val_users=rank_user_pool[n_rank_train:]
rank_train_events=rank_source[rank_source.user_id.isin(rank_train_users)].copy()
rank_val_events=rank_source[rank_source.user_id.isin(rank_val_users)].copy()
all_positive_by_user={u:set(g.item_id) for u,g in positive.groupby('user_id')}
read_values=pd.to_numeric(rank_source.read_percentage,errors='coerce').dropna()
READ_THRESHOLD=0.8 if len(read_values)==0 or read_values.quantile(.95)<=1.5 else 80.0
item_pop=train_df.groupby('item_id').size().reindex(item_ids,fill_value=0).to_numpy(np.float32)
item_pop=np.log1p(item_pop); item_pop=(item_pop-item_pop.mean())/(item_pop.std()+1e-6)

def label_lookup(events):
    x=events.copy()
    for c in ['like','share','follow','favorite','read_percentage']:
        x[c]=pd.to_numeric(x[c],errors='coerce').fillna(0)
    x['long_read']=(x.read_percentage>=READ_THRESHOLD).astype('float32')
    x=(x.groupby(['user_id','item_id'],as_index=False,sort=False)
       .agg(like=('like','max'),share=('share','max'),follow=('follow','max'),favorite=('favorite','max'),long_read=('long_read','max')))
    return {(r.user_id,r.item_id):r for r in x.itertuples()}

def build_rank_frame(users, target_events, training, route_cache=None):
    labels=label_lookup(target_events); target_by_user={}
    for (u,i) in labels: target_by_user.setdefault(u,set()).add(i)
    rows=[]; local_rng=np.random.default_rng(RANK_SEED + int(training))
    for uid in tqdm(users,desc='构造排序训练候选' if training else '构造排序验证候选'):
        if uid not in u2i: continue
        routes=(route_cache[uid] if route_cache is not None else {name:recallers[name](uid) for name in SOURCE_ORDER})
        rank_maps={name:{item:rank for rank,item in enumerate(routes[name],start=1)} for name in SOURCE_ORDER}
        pool=list(dict.fromkeys(item for name in SOURCE_ORDER for item in routes[name] if item in it2i))
        positives=[i for i in target_by_user.get(uid,set()) if i in it2i]
        if training:
            negative_pool=[i for i in pool if i not in all_positive_by_user.get(uid,set())]
            n_neg=min(len(negative_pool),NEGATIVES_PER_POSITIVE*max(1,len(positives)))
            negatives=local_rng.choice(negative_pool,size=n_neg,replace=False).tolist() if n_neg else []
            selected=list(dict.fromkeys(positives+negatives))
        else:
            # 用完整正反馈集合只清洗假负样本，不把未召回的目标注入候选。
            selected=[i for i in pool if i not in all_positive_by_user.get(uid,set()) or i in target_by_user.get(uid,set())]
        for iid in selected:
            lab=labels.get((uid,iid)); rec={'user_id':uid,'item_id':iid,'user_idx':u2i[uid],'item_idx':it2i[iid]}
            rec['relevance']=float(lab is not None)
            for task in ['like','share','follow','favorite','long_read']:
                rec[task]=float(getattr(lab,task,0)>0) if lab is not None else 0.0
            for name in SOURCE_ORDER:
                rank=rank_maps[name].get(iid,0); rec[f'{name}_hit']=float(rank>0); rec[f'{name}_rr']=1.0/rank if rank else 0.0
            rec['source_count']=sum(rec[f'{name}_hit'] for name in SOURCE_ORDER)
            rec['rrf_score']=sum(1.0/(60+rank_maps[name][iid]) for name in SOURCE_ORDER if iid in rank_maps[name])
            rec['item_pop_z']=float(item_pop[it2i[iid]])
            rows.append(rec)
    return pd.DataFrame(rows), target_by_user

rank_train_frame,rank_train_targets=build_rank_frame(rank_train_users,rank_train_events,training=True)
rank_val_frame,rank_val_targets=build_rank_frame(rank_val_users,rank_val_events,training=False)
rank_train_frame=rank_train_frame.reset_index(drop=True); rank_val_frame=rank_val_frame.reset_index(drop=True)
rank_train_frame.to_parquet(DATA_DIR/'rank_train_candidates.parquet',index=False)
rank_val_frame.to_parquet(DATA_DIR/'rank_val_candidates.parquet',index=False)
print({'rank_train_rows':len(rank_train_frame),'rank_train_positive_rate':rank_train_frame.relevance.mean(),'rank_val_rows':len(rank_val_frame),'rank_val_positive_rate':rank_val_frame.relevance.mean(),'read_threshold':READ_THRESHOLD})

## 7. DCNv2 粗排与 DCNv2+MMoE 精排

Student 使用小型 DCNv2；Teacher 使用更大的 DCNv2 shared bottom，再接 MMoE experts/gates。两者都复用召回双塔已经学到的 user/item embedding 作为初始化。

In [ ]:
RANK_NUM_COLS=[f'{name}_hit' for name in SOURCE_ORDER]+[f'{name}_rr' for name in SOURCE_ORDER]+['source_count','rrf_score','item_pop_z']
rank_num_mean=rank_train_frame[RANK_NUM_COLS].mean().to_numpy(np.float32)
rank_num_std=rank_train_frame[RANK_NUM_COLS].std().fillna(0).to_numpy(np.float32)+1e-6
def frame_tensors(frame):
    nums=(frame[RANK_NUM_COLS].to_numpy(dtype=np.float32,copy=True)-rank_num_mean)/rank_num_std
    labels=frame[list(TASKS)].to_numpy(dtype=np.float32,copy=True)
    return (torch.tensor(frame.user_idx.to_numpy(np.int64)),torch.tensor(frame.item_idx.to_numpy(np.int64)),torch.tensor(nums),torch.tensor(labels))
train_tensors=frame_tensors(rank_train_frame); val_tensors=frame_tensors(rank_val_frame)
train_loader=DataLoader(TensorDataset(*train_tensors),batch_size=RANK_BATCH_SIZE,shuffle=True,num_workers=0,pin_memory=torch.cuda.is_available())
val_loader=DataLoader(TensorDataset(*val_tensors),batch_size=RANK_BATCH_SIZE,shuffle=False,num_workers=0)
recall_user_init=model.ue.weight.detach().clone(); recall_item_init=model.ie.weight.detach().clone()

class RankFeatureEncoder(nn.Module):
    def __init__(self):
        super().__init__(); self.ue=nn.Embedding.from_pretrained(recall_user_init.clone(),freeze=False); self.ie=nn.Embedding.from_pretrained(recall_item_init.clone(),freeze=False)
        self.ge=nn.Embedding(len(gmap)+1,4); self.ae=nn.Embedding(len(amap)+1,4); self.c1e=nn.Embedding(len(c1map)+1,8); self.c2e=nn.Embedding(len(c2map)+1,8)
        self.output_dim=2*LATENT_DIM+4+4+8+8+6+3+len(RANK_NUM_COLS)
    def forward(self,u,i,xnum):
        return torch.cat([self.ue(u),self.ie(i),self.ge(ug[u]),self.ae(ua[u]),self.c1e(ic1[i]),self.c2e(ic2[i]),user_numeric[u],quality[i],xnum],dim=-1)
class CrossLayerV2(nn.Module):
    def __init__(self,d): super().__init__(); self.proj=nn.Linear(d,d)
    def forward(self,x0,x): return x0*self.proj(x)+x
class DCNBackbone(nn.Module):
    def __init__(self,cross_layers=3,deep=(256,128),out_dim=128):
        super().__init__(); self.encoder=RankFeatureEncoder(); d=self.encoder.output_dim; self.cross=nn.ModuleList([CrossLayerV2(d) for _ in range(cross_layers)])
        layers=[]; cur=d
        for h in deep: layers += [nn.Linear(cur,h),nn.ReLU(),nn.Dropout(.1)]; cur=h
        self.deep=nn.Sequential(*layers); self.out=nn.Sequential(nn.Linear(d+cur,out_dim),nn.ReLU())
    def forward(self,u,i,xnum):
        x0=self.encoder(u,i,xnum); cross=x0
        for layer in self.cross: cross=layer(x0,cross)
        return self.out(torch.cat([cross,self.deep(x0)],dim=-1))
class DCNStudent(nn.Module):
    def __init__(self): super().__init__(); self.backbone=DCNBackbone(3,(256,128),128); self.head=nn.Linear(128,1)
    def forward(self,u,i,xnum): return self.head(self.backbone(u,i,xnum)).squeeze(-1)
class DCNMMoETeacher(nn.Module):
    def __init__(self,n_experts=6,expert_dim=96):
        super().__init__(); self.backbone=DCNBackbone(4,(384,256,128),192)
        self.experts=nn.ModuleList([nn.Sequential(nn.Linear(192,192),nn.ReLU(),nn.Dropout(.1),nn.Linear(192,expert_dim),nn.ReLU()) for _ in range(n_experts)])
        self.gates=nn.ModuleList([nn.Linear(192,n_experts) for _ in TASKS]); self.towers=nn.ModuleList([nn.Sequential(nn.Linear(expert_dim,64),nn.ReLU(),nn.Linear(64,1)) for _ in TASKS])
    def forward(self,u,i,xnum):
        h=self.backbone(u,i,xnum); experts=torch.stack([e(h) for e in self.experts],dim=1); outputs=[]
        for gate,tower in zip(self.gates,self.towers):
            mixed=(experts*torch.softmax(gate(h),dim=-1).unsqueeze(-1)).sum(1); outputs.append(tower(mixed).squeeze(-1))
        return torch.stack(outputs,dim=1)
print({'rank_numeric_features':RANK_NUM_COLS,'student_input_dim':RankFeatureEncoder().output_dim,'tasks':TASKS})

## 8. 训练 MMoE Teacher

`relevance` 是候选是否属于排序目标区间的点击文章；其余任务来自该正样本的强反馈。对隐式负样本，所有任务标签为 0。稀有任务使用 capped `pos_weight`。

In [ ]:
def safe_auc(y,s):
    return float(roc_auc_score(y,s)) if len(np.unique(y))>1 else np.nan
@torch.no_grad()
def predict_teacher(net,loader):
    net.eval(); out=[]
    for u,i,x,y in loader: out.append(net(u.to(DEVICE),i.to(DEVICE),x.to(DEVICE)).cpu())
    return torch.cat(out).numpy()
teacher=DCNMMoETeacher().to(DEVICE); teacher_opt=torch.optim.AdamW(teacher.parameters(),lr=1e-3,weight_decay=1e-5)
label_sum=train_tensors[3].sum(0); task_pos_weight=((len(rank_train_frame)-label_sum)/(label_sum+1)).clamp(1,50).to(DEVICE)
best_teacher_auc=-np.inf; best_teacher_path=MODEL_DIR/'dcnv2_mmoe_teacher.pt'
for epoch in range(TEACHER_EPOCHS):
    teacher.train(); total=0.; count=0
    progress=tqdm(train_loader,desc=f'Teacher {epoch+1}/{TEACHER_EPOCHS}')
    for u,i,x,y in progress:
        u,i,x,y=u.to(DEVICE),i.to(DEVICE),x.to(DEVICE),y.to(DEVICE); logits=teacher(u,i,x)
        loss=F.binary_cross_entropy_with_logits(logits,y,pos_weight=task_pos_weight)
        teacher_opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(teacher.parameters(),5.0); teacher_opt.step(); total+=loss.item()*len(u); count+=len(u)
        progress.set_postfix(loss=f'{total/count:.4f}')
    val_logits=predict_teacher(teacher,val_loader); val_auc=safe_auc(val_tensors[3][:,0].numpy(),val_logits[:,0])
    print({'epoch':epoch+1,'teacher_loss':total/count,'val_relevance_auc':val_auc})
    if epoch==0 or (np.isfinite(val_auc) and val_auc>best_teacher_auc): best_teacher_auc=val_auc; torch.save(teacher.state_dict(),best_teacher_path)
teacher.load_state_dict(torch.load(best_teacher_path,map_location=DEVICE,weights_only=True))
teacher_val_logits=predict_teacher(teacher,val_loader)
teacher_task_report=pd.DataFrame({'auc':[safe_auc(val_tensors[3][:,j].numpy(),teacher_val_logits[:,j]) for j in range(len(TASKS))], 'pr_auc':[float(average_precision_score(val_tensors[3][:,j].numpy(),teacher_val_logits[:,j])) if val_tensors[3][:,j].sum()>0 else np.nan for j in range(len(TASKS))]},index=TASKS)
display(teacher_task_report)

## 9. Teacher → DCNv2 Student 蒸馏

Student 总损失由三部分组成：硬标签 BCE、Teacher 融合分数的 Huber loss、Teacher 排序产生的 RankNet pairwise loss。这里不做表征蒸馏。

In [ ]:
UTILITY_WEIGHTS=torch.tensor([1.0,3.0,4.0,4.0,5.0,1.5],device=DEVICE)
DISTILL_ALPHA,DISTILL_BETA,DISTILL_GAMMA=1.0,.5,.2
@torch.no_grad()
def teacher_utility(net,loader):
    net.eval(); out=[]
    for u,i,x,y in loader:
        logits=net(u.to(DEVICE),i.to(DEVICE),x.to(DEVICE)); out.append((torch.sigmoid(logits)*UTILITY_WEIGHTS).sum(1).cpu())
    return torch.cat(out)
ordered_train_loader=DataLoader(TensorDataset(*train_tensors),batch_size=RANK_BATCH_SIZE,shuffle=False,num_workers=0)
teacher_train_score=teacher_utility(teacher,ordered_train_loader)
teacher_score_mean,teacher_score_std=teacher_train_score.mean(),teacher_train_score.std().clamp_min(1e-6)
teacher_train_target=(teacher_train_score-teacher_score_mean)/teacher_score_std
# 每个用户从 Teacher 高分与低分候选间采样排序对。
pair_left=[]; pair_right=[]; pair_rng=np.random.default_rng(RANK_SEED)
for idx in rank_train_frame.groupby('user_id',sort=False).indices.values():
    idx=np.asarray(idx);
    if len(idx)<2: continue
    order=idx[np.argsort(teacher_train_score[idx].numpy())]; low=order[:min(5,len(order)//2)]; high=order[-min(5,len(order)//2):]
    for _ in range(min(5,len(low)*len(high))): pair_left.append(int(pair_rng.choice(high))); pair_right.append(int(pair_rng.choice(low)))
pair_left=torch.tensor(pair_left,dtype=torch.long); pair_right=torch.tensor(pair_right,dtype=torch.long)
distill_loader=DataLoader(TensorDataset(*train_tensors,teacher_train_target),batch_size=RANK_BATCH_SIZE,shuffle=True,num_workers=0,pin_memory=torch.cuda.is_available())
student=DCNStudent().to(DEVICE); student_opt=torch.optim.AdamW(student.parameters(),lr=1e-3,weight_decay=1e-5)
student_pos_weight=task_pos_weight[0]
best_student_auc=-np.inf; best_student_path=MODEL_DIR/'dcnv2_coarse_student.pt'
for epoch in range(STUDENT_EPOCHS):
    student.train(); totals=np.zeros(4); count=0; progress=tqdm(distill_loader,desc=f'Student {epoch+1}/{STUDENT_EPOCHS}')
    for u,i,x,y,t_score in progress:
        u,i,x,y,t_score=u.to(DEVICE),i.to(DEVICE),x.to(DEVICE),y.to(DEVICE),t_score.to(DEVICE); s=student(u,i,x)
        hard=F.binary_cross_entropy_with_logits(s,y[:,0],pos_weight=student_pos_weight); score_loss=F.smooth_l1_loss(s,t_score)
        pidx=torch.randint(len(pair_left),(len(u),)); left,right=pair_left[pidx],pair_right[pidx]
        sl=student(train_tensors[0][left].to(DEVICE),train_tensors[1][left].to(DEVICE),train_tensors[2][left].to(DEVICE))
        sr=student(train_tensors[0][right].to(DEVICE),train_tensors[1][right].to(DEVICE),train_tensors[2][right].to(DEVICE)); rank_loss=F.softplus(-(sl-sr)).mean()
        loss=DISTILL_ALPHA*hard+DISTILL_BETA*score_loss+DISTILL_GAMMA*rank_loss
        student_opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(student.parameters(),5.0); student_opt.step()
        totals+=np.array([loss.item(),hard.item(),score_loss.item(),rank_loss.item()])*len(u); count+=len(u); progress.set_postfix(total=f'{totals[0]/count:.4f}')
    student.eval(); val_score=[]
    with torch.no_grad():
        for vu,vi,vx,vy in val_loader: val_score.append(student(vu.to(DEVICE),vi.to(DEVICE),vx.to(DEVICE)).cpu())
    val_score=torch.cat(val_score).numpy(); val_auc=safe_auc(rank_val_frame.relevance.to_numpy(),val_score)
    print({'epoch':epoch+1,'total':totals[0]/count,'hard':totals[1]/count,'score_distill':totals[2]/count,'rank_distill':totals[3]/count,'val_relevance_auc':val_auc})
    if epoch==0 or (np.isfinite(val_auc) and val_auc>best_student_auc): best_student_auc=val_auc; torch.save(student.state_dict(),best_student_path)
student.load_state_dict(torch.load(best_student_path,map_location=DEVICE,weights_only=True))

## 10. 排序与蒸馏指标

粗排在完整候选池上取 Top-100；精排只处理这 100 条并取 Top-20。同时给出 Teacher 若直接看完整候选池时的上界，以及 Student/Teacher Top-100 重合率。

In [ ]:
@torch.no_grad()
def predict_student(net,tensors,batch=8192):
    net.eval(); out=[]
    loader=DataLoader(TensorDataset(*tensors),batch_size=batch,shuffle=False,num_workers=0)
    for u,i,x,y in loader: out.append(net(u.to(DEVICE),i.to(DEVICE),x.to(DEVICE)).cpu())
    return torch.cat(out).numpy()
def dcg(relevance): return sum(float(r)/np.log2(k+2) for k,r in enumerate(relevance))
def evaluate_pipeline(frame,targets,student_score,teacher_score):
    coarse_recall=[]; fine_recall=[]; teacher_full_recall=[]; fine_ndcg=[]; overlap=[]; groups=frame.groupby('user_id',sort=False).indices
    for uid,target_items in targets.items():
        truth=set(target_items); idx=np.asarray(groups.get(uid,[]),dtype=np.int64)
        if not truth: continue
        if len(idx)==0:
            coarse_recall.append(0.); fine_recall.append(0.); teacher_full_recall.append(0.); fine_ndcg.append(0.); overlap.append(0.); continue
        s_order=idx[np.argsort(-student_score[idx])]; t_order=idx[np.argsort(-teacher_score[idx])]
        coarse=s_order[:COARSE_TOPK]; final=coarse[np.argsort(-teacher_score[coarse])][:FINE_TOPK]; teacher_full=t_order[:FINE_TOPK]
        coarse_items=frame.item_id.iloc[coarse].to_list(); final_items=frame.item_id.iloc[final].to_list(); full_items=frame.item_id.iloc[teacher_full].to_list()
        coarse_recall.append(len(set(coarse_items)&truth)/len(truth)); fine_recall.append(len(set(final_items)&truth)/len(truth)); teacher_full_recall.append(len(set(full_items)&truth)/len(truth))
        rel=[int(x in truth) for x in final_items]; ideal=[1]*min(len(truth),FINE_TOPK); fine_ndcg.append(dcg(rel)/(dcg(ideal) or 1))
        overlap.append(len(set(frame.item_id.iloc[s_order[:COARSE_TOPK]])&set(frame.item_id.iloc[t_order[:COARSE_TOPK]]))/max(1,min(COARSE_TOPK,len(idx))))
    return {'coarse_recall@100':np.mean(coarse_recall),'fine_recall@20':np.mean(fine_recall),'teacher_full_recall@20':np.mean(teacher_full_recall),'fine_ndcg@20':np.mean(fine_ndcg),'student_teacher_top100_overlap':np.mean(overlap)}
student_val_score=predict_student(student,val_tensors)
teacher_val_score=(torch.sigmoid(torch.tensor(teacher_val_logits))*UTILITY_WEIGHTS.cpu()).sum(1).numpy()
val_pipeline_report=evaluate_pipeline(rank_val_frame,rank_val_targets,student_val_score,teacher_val_score)
val_pipeline_report.update({'student_candidate_auc':safe_auc(rank_val_frame.relevance.to_numpy(),student_val_score),'teacher_candidate_auc':safe_auc(rank_val_frame.relevance.to_numpy(),teacher_val_score)})
display(pd.DataFrame([val_pipeline_report],index=['rank_validation']))

# 最终 10% 测试：直接复用第 5 节已经生成的四路候选，不注入测试正样本。
test_route_cache={u:{name:results[name][u] for name in SOURCE_ORDER} for u in eval_targets}
test_rank_events=test_events[test_events.user_id.isin(eval_targets) & test_events.item_id.isin(it2i)].copy()
test_rank_events=test_rank_events[[row.item_id not in seen.get(row.user_id,set()) for row in test_rank_events.itertuples()]]
test_rank_frame,_=build_rank_frame(list(eval_targets),test_rank_events,training=False,route_cache=test_route_cache)
test_rank_frame=test_rank_frame.reset_index(drop=True); test_tensors=frame_tensors(test_rank_frame)
student_test_score=predict_student(student,test_tensors)
test_teacher_logits=predict_teacher(teacher,DataLoader(TensorDataset(*test_tensors),batch_size=RANK_BATCH_SIZE,shuffle=False,num_workers=0))
teacher_test_score=(torch.sigmoid(torch.tensor(test_teacher_logits))*UTILITY_WEIGHTS.cpu()).sum(1).numpy()
test_pipeline_report=evaluate_pipeline(test_rank_frame,eval_targets,student_test_score,teacher_test_score)
test_pipeline_report.update({'student_candidate_auc':safe_auc(test_rank_frame.relevance.to_numpy(),student_test_score),'teacher_candidate_auc':safe_auc(test_rank_frame.relevance.to_numpy(),teacher_test_score)})
final_report=pd.DataFrame([val_pipeline_report,test_pipeline_report],index=['rank_validation','final_test'])
display(final_report)
test_rank_frame.assign(student_score=student_test_score,teacher_score=teacher_test_score).to_parquet(RESULT_DIR/'dcn_mmoe_scored_test_candidates.parquet',index=False)
teacher_task_report.to_csv(RESULT_DIR/'mmoe_teacher_task_metrics.csv'); final_report.to_csv(RESULT_DIR/'dcn_mmoe_pipeline_metrics.csv')
print('排序模型与结果已保存到',RESULT_DIR)